# FIA Regulation RAG

Trabalho da disciplina de Inteligência Artificial da **UFCG**, com o **Prof. Herman Martins Gomes**, encerrado em **05/03/2026**.

**Autores:** Maria Clara ([@claraugusta](https://github.com/claraugusta)), Gledson Luan ([@GledsonLScruz](https://github.com/GledsonLScruz)), Arthur Félix ([@ArthurHappx](https://github.com/ArthurHappx)), Emmily Almeida e Luis Alberto.

[Apresentação](https://canva.link/awqly4iidizn969) · [Documentação](README.md) · [Resultados históricos e limitações](results/README.md) · [Google Colab com execução original](https://colab.research.google.com/drive/1yZ3uLqYBmWNSciBQEIDrdaMXlu_8RUdU?usp=sharing)

O pipeline compara três estratégias de pré-processamento dos regulamentos FIA 2026: **A (Standard)**, Markdown dividido em chunks; **B (Fine grained)**, itens estruturados com metadados; **C (Coarse grained)**, cláusulas extraídas do Markdown. Há também um experimento sem recuperação.

A avaliação usa cinco critérios inspirados no artigo [CCRS, de Aashiq Muhamed (2025)](https://arxiv.org/abs/2506.20128). Esta é uma adaptação ao domínio FIA, sem pretensão de reproduzir integralmente os experimentos do artigo.

**Versão de portfólio:** a execução local e os modelos de API foram atualizados após o encerramento do trabalho. O prompt sem RAG e a validação das notas foram corrigidos para novas execuções. Os resultados históricos em `results/historical/` não foram recalculados e não devem ser confundidos com os modelos atuais. Para checar os resultados obtidos originalmente, consulte [Google Colab com execução original](https://colab.research.google.com/drive/1yZ3uLqYBmWNSciBQEIDrdaMXlu_8RUdU?usp=sharing).

# 1. Configuração do ambiente local

No VS Code, selecione `.venv/Scripts/python.exe` como kernel. As dependências estão em `requirements.txt`; consulte o `README.md` para preparar o ambiente.

A configuração abaixo baixa os dados somente se necessário e usa caminhos relativos à pasta do projeto. Os modelos são executados em CPU; a geração e tradução usam APIs externas.

In [ ]:
import sys
import subprocess
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "FIA_Regulation_RAG.ipynb").is_file():
    raise RuntimeError("Abra o notebook a partir da pasta FIA_Regulation_RAG.")
if not ((3, 11) <= sys.version_info[:2] <= (3, 14)):
    raise RuntimeError("Selecione o kernel Python 3.12 de .venv/Scripts/python.exe.")

DATA_REVISION = "b7fc835a3ebd080a919aa3cc8a531c667b7e90a2"
DATA_DIR = PROJECT_ROOT / "filesIAColab"
if not DATA_DIR.exists():
    subprocess.run([
        "git", "clone", "--no-checkout", "--depth", "1",
        "https://github.com/GledsonLScruz/filesIAColab.git", str(DATA_DIR),
    ], check=True)
    subprocess.run(["git", "-C", str(DATA_DIR), "fetch", "--depth", "1", "origin", DATA_REVISION], check=True)
    subprocess.run(["git", "-C", str(DATA_DIR), "checkout", "--detach", DATA_REVISION], check=True)

revision_check = subprocess.run(
    ["git", "-C", str(DATA_DIR), "rev-parse", "HEAD"],
    capture_output=True, text=True,
)
DATA_ACTUAL_REVISION = revision_check.stdout.strip() if revision_check.returncode == 0 else None
if DATA_ACTUAL_REVISION != DATA_REVISION:
    print("AVISO: revisão dos dados diferente ou não verificável; os resultados podem variar.")
if DATA_ACTUAL_REVISION:
    status_check = subprocess.run(
        ["git", "-C", str(DATA_DIR), "status", "--porcelain"], capture_output=True, text=True,
    )
    if status_check.returncode != 0 or status_check.stdout.strip():
        print("AVISO: há alterações locais nos dados ou não foi possível verificar seu estado.")

required_files = [
    f"CLEAN_FIA_2026_F1_Regulations_{letter}_[{name}].md"
    for letter, name in [("A", "General"), ("B", "Sporting"),
                         ("C", "Technical"), ("F", "Operational")]
] + ["fia_rag_chunks.jsonl", "tests.json", "prompts/llm_judges_prompts.py"]
missing_files = [name for name in required_files if not (DATA_DIR / name).is_file()]
if missing_files:
    raise FileNotFoundError(f"Dados ausentes em {DATA_DIR}: {missing_files}")
if str(DATA_DIR) not in sys.path:
    sys.path.insert(0, str(DATA_DIR))

print(f"Python: {sys.version.split()[0]}")
print(f"Kernel: {sys.executable}")
print(f"Dados: {DATA_DIR}")

# 2. Chaves e configurações

Carrega o `.env` local sem exibir os valores. NVIDIA gera as respostas e executa os juízes; DeepL traduz as perguntas. A chave OpenAI não é usada nesta versão.

As avaliações em lote ficam desativadas até definir `RUN_EVALUATIONS = True`.

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv(PROJECT_ROOT / ".env")
os.environ.setdefault("HF_HOME", str(PROJECT_ROOT / ".cache" / "huggingface"))
os.environ.setdefault("MPLCONFIGDIR", str(PROJECT_ROOT / ".cache" / "matplotlib"))
missing_keys = [name for name in ("NVIDIA_API_KEY", "DEEPL_API_KEY")
                if not os.environ.get(name, "").strip()]
if missing_keys:
    raise RuntimeError("Preencha no .env: " + ", ".join(missing_keys))

RUN_EVALUATIONS = False
NVIDIA_MODEL = os.getenv("NVIDIA_MODEL", "nvidia/nemotron-3.5-lightning-30b-a3b")
NVIDIA_JUDGE_MODEL = os.getenv("NVIDIA_JUDGE_MODEL", "openai/gpt-oss-20b")
print("Chaves NVIDIA e DeepL carregadas. Valores ocultos.")
print(f"Avaliações em lote: {RUN_EVALUATIONS}")

# 3. Método (A) Standard RAG: Markdown bruto + chunking

Cada método evocou um tipo de pré-processamento (data cleaning) e um tipo de coleta de metadados usando pattern matching (sobretudo com regex). A fim de deixar o colab mais suscinto, alguns desses script foram omitidos e adicionados no repositório base para consulta [filesIAColab](https://github.com/GledsonLScruz/filesIAColab.git).

Carrega os arquivos `CLEAN_*.md` e aplica chunking simples por tamanho.
Esse método é usado como baseline.

In [ ]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
import os
import re

# Lista de documentos Markdown limpos
documents = [
    "filesIAColab/CLEAN_FIA_2026_F1_Regulations_A_[General].md",
    "filesIAColab/CLEAN_FIA_2026_F1_Regulations_B_[Sporting].md",
    "filesIAColab/CLEAN_FIA_2026_F1_Regulations_C_[Technical].md",
    "filesIAColab/CLEAN_FIA_2026_F1_Regulations_F_[Operational].md",
]

all_docs = []

def remove_space(texto):
    return re.sub(r'[ 	]+', ' ', texto)

def save_documents(documents, all_docs):
    for document in documents:
        if not os.path.exists(document):
            print(f"ERRO: O arquivo '{document}' nao foi encontrado!")
            continue
        loader = TextLoader(document, encoding="utf-8")
        docs = loader.load()
        for doc in docs:
            doc.page_content = remove_space(doc.page_content)
            doc.metadata["source_file"] = document
            all_docs.append(doc)

# Carrega documentos
save_documents(documents, all_docs)

# Chunking simples
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1200,
    chunk_overlap=150,
    separators=["\n## ","\n### ","\n[A-Z]\\d\\.","\n\n","\n"," "],
    is_separator_regex=True,
)

chunks_method_a = text_splitter.split_documents(all_docs)
print(f"Metodo A -> {len(chunks_method_a)} chunks")


# 4. Método (B) Fine Grained: JSONL estruturado

Este método usa o preprocessamento com metadados de clausulas e itens.

In [ ]:
import json
from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings

documents_to_index = []

def load_documents_from_jsonl(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            data = json.loads(line)
            doc = Document(
                page_content=data['text'],
                metadata=data['metadata'],
            )
            documents_to_index.append(doc)

# Use o arquivo gerado por preprocess_fia_v2.py
load_documents_from_jsonl("filesIAColab/fia_rag_chunks.jsonl")

print(f"Metodo B -> {len(documents_to_index)} documentos estruturados")


# 5. Metodo (C) Coarse Grained: Extração Granular por Cláusulas

Ele lê o arquivo Markdown e cria "mapas" de memória para identificar o que é Artigo (Ex: ARTICLE B3) e o que é Subtítulo. Isso garante que cada frase do regulamento saiba exatamente a qual título pertence.


In [ ]:
glossary_docs = []

# Usar o glossario gerado por preprocess_fia_v2.py
# load_documents_from_jsonl("data-set/fia_rag_glossary.jsonl")
# glossary_docs = documents_to_index

# Alternativa: carregar em lista separada
# (manter docs de glossario separados de docs de clausulas)
import re
import json
import os
from langchain_core.documents import Document

def extract_fia_rag_final(files):
    final_jsonl = []

    for file in files:
        titles_map = {}
        subtitles_map = {}

        with open(file, "r", encoding="utf-8") as f:
            markdown_input = f.read()

        source_name = os.path.basename(file)

        # Detecta qual regulamento é (A,B,C,F)
        reg_letter = source_name.split("_Regulations_")[1][0]

        # Nome da seção
        section_map = {
            "A": "General Regulations",
            "B": "Sporting Regulations",
            "C": "Technical Regulations",
            "F": "Operational Regulations"
        }

        section_name = section_map.get(reg_letter, "Unknown")

        lines = markdown_input.split('\n')

        # Mapeia títulos e sub-títulos
        for line in lines:
            line = line.strip()

            article_match = re.search(r'##\s+ARTICLE\s+([A-Z]\d+):\s*(.*)', line)
            if article_match:
                titles_map[article_match.group(1)] = article_match.group(2).strip()
                continue

            sub_match = re.match(r'^([A-Z]\d+\.\d+)\s+(.*)', line)
            if sub_match:
                subtitles_map[sub_match.group(1)] = sub_match.group(2).strip()

        # Localiza conteúdo real
        content_start = markdown_input.find(f"## ARTICLE {reg_letter}3", 100)
        real_content = markdown_input[content_start:]

        # Divide por artigos
        articles = re.split(r'\n(?=##\s+ARTICLE\s+[A-Z])', real_content)

        for article in articles:
            article = article.strip()
            if not article:
                continue

            art_id_match = re.search(r'##\s+ARTICLE\s+([A-Z]\d+)', article)
            if not art_id_match:
                continue

            art_id = art_id_match.group(1)
            current_title = titles_map.get(art_id, "Unknown Title")

            # Divide por cláusulas
            clauses = re.split(r'\n(?=[A-Z]\d+\.\d+\.\d+)', article)

            for clause_text in clauses:
                clause_text = clause_text.strip()

                if not clause_text or clause_text.startswith('##'):
                    continue

                match = re.match(r'^([A-Z])((\d+\.\d+)\.(\d+))', clause_text)

                if match:
                    reg_letter = match.group(1)
                    sub_id_num = match.group(3)
                    topic_num = match.group(4)

                    full_sub_id = reg_letter + sub_id_num
                    clause_id = f"{reg_letter}{sub_id_num}.{topic_num}"

                    current_subtitle = subtitles_map.get(full_sub_id, "Unknown Subtitle")

                    clean_text = " ".join(clause_text.split())

                    entry = {
                        "text": clean_text,
                        "metadata": {
                            "title": current_title,
                            "subtitle": current_subtitle,
                            "subarticle": sub_id_num,
                            "topic": topic_num,
                            "clause_id": clause_id,
                            "regulation_type": section_name,
                            "source_doc": source_name
                        }
                    }

                    final_jsonl.append(entry)

    return final_jsonl


files = [
    "filesIAColab/CLEAN_FIA_2026_F1_Regulations_A_[General].md",
    "filesIAColab/CLEAN_FIA_2026_F1_Regulations_B_[Sporting].md",
    "filesIAColab/CLEAN_FIA_2026_F1_Regulations_C_[Technical].md",
    "filesIAColab/CLEAN_FIA_2026_F1_Regulations_F_[Operational].md",
]

jsonl_data = extract_fia_rag_final(files)
documents_method_c = []

for item in jsonl_data:
    doc = Document(
        page_content=item["text"],
        metadata=item["metadata"]
    )
    documents_method_c.append(doc)

print(f"Método C -> {len(documents_method_c)} documentos convertidos para LangChain")


# 6. Embeddings e Vector Stores

Usamos `all-MiniLM-L6-v2` por ser leve e rápido.
Criaremos um FAISS para cada método, permitindo comparar impacto do preprocessamento.

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

embeddings_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2", model_kwargs={"device": "cpu"})

# Metodo Standard Chunks
vector_store_a = FAISS.from_documents(chunks_method_a, embeddings_model)
vector_store_a.save_local("vector_store_method_a")

# Metodo fine grained
vector_store_b = FAISS.from_documents(documents_to_index, embeddings_model)
vector_store_b.save_local("vector_store_method_b")

# Metodo Coarse grained
vector_store_c = FAISS.from_documents(documents_method_c, embeddings_model)
vector_store_c.save_local("vector_store_method_c")

print("Vector stores criadas para Metodo A, B e C")

# 7. Traducao da query (PT -> EN)

O `RAG.ipynb` usa DeepL para traduzir a pergunta antes da busca vetorial.
Isso melhora o recall já que o indice está em ingles, mas pode perder termos técnicos.


In [ ]:
import deepl

_deepl_key = os.environ.get("DEEPL_API_KEY", "")
if _deepl_key:
    deepl_client = deepl.DeepLClient(_deepl_key)
else:
    deepl_client = None


def deepl_translate(source: str) -> str:
    if not deepl_client:
        raise RuntimeError("deepl_client não encontrado")
    result = deepl_client.translate_text(source, target_lang="EN-US")
    return result.text


# 8. Reranking (cross-encoder)

O cross-encoder melhora a precisão reordenando os documentos recuperados.

In [ ]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2", device="cpu")


def rerank_documents(query, docs, top_k=5):
    pairs = [[query, doc.page_content] for doc in docs]
    scores = reranker.predict(pairs)
    docs_with_scores = list(zip(docs, scores))
    docs_sorted = sorted(docs_with_scores, key=lambda x: x[1], reverse=True)
    reranked_docs = [doc for doc, score in docs_sorted[:top_k]]
    return reranked_docs

# 9. LLM de resposta

Aqui usamos NVIDIA endpoints.

In [ ]:
# Consulta opcional: execute listar_modelos_nvidia() para visualizar o catálogo.
from langchain_nvidia_ai_endpoints import ChatNVIDIA
import pandas as pd

def listar_modelos_nvidia():
    nv_client = ChatNVIDIA(model=NVIDIA_MODEL, api_key=os.environ["NVIDIA_API_KEY"])
    models = nv_client.available_models
    return pd.DataFrame([
        {"id": model.id, "type": getattr(model, "model_type", "N/A")}
        for model in models
    ]).sort_values(by="id")

In [ ]:
from langchain_nvidia_ai_endpoints import ChatNVIDIA

client = ChatNVIDIA(
    model=NVIDIA_MODEL,
    api_key=os.environ.get("NVIDIA_API_KEY", ""),
    temperature=0.5,
    top_p=0.7,
    max_completion_tokens=2048,
)

# 10. Prompt de geracao

Responde usando somente contexto recuperado, com tom técnico e em PT-BR.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
         ("system", """You are an AI assistant specialized in analyzing official Formula 1 regulations.

 You must answer using ONLY the provided context.
 Do not use prior knowledge or external information.

 PROCESS:
 - Identify the passages in the context that are relevant to the question.
 - Quote or reference the relevant article, section, or paragraph.
 - Based only on those passages, reason about the rule and produce the final answer.

 RULES:
 - The answer may combine or logically infer information from multiple passages in the context.
 - Every conclusion must be supported by a cited article, section, or paragraph.
 - Do not invent rules or information not present in the context.
 - If the information is not present or cannot be inferred from the context, respond exactly with: "Não foi possível encontrar essa informacao no documento fornecido".

 STYLE:
 - Final explanation of the rule based strictly on the evidence.
 - Maintain a formal and technical tone.
 - Be precise and concise.
 - Write the final answer in Brazilian Portuguese.
 """), ("user", "Context: {context} Question:{query}"),
 ])

chain = prompt | client

# Baseline independente: sem contexto recuperado e sem a restrição do prompt RAG.
prompt_no_rag = ChatPromptTemplate.from_messages([
    ("system", """You answer questions about Formula 1 regulations using your own knowledge.
No reference documents are supplied. Answer precisely and concisely in Brazilian Portuguese.
If you do not know the answer, state your uncertainty. Do not invent regulations or citations."""),
    ("user", "{query}"),
])
chain_no_rag = prompt_no_rag | client


def generate_answer(query, retrieved_docs, use_rag=True):
    if not use_rag:
        return chain_no_rag.invoke({"query": query}).content
    context = "\n\n".join([doc.page_content for doc in retrieved_docs])
    response = chain.invoke({"context": context, "query": query})
    return response.content

# 11. Recuperação + Geração

Executa a busca no FAISS, faz rerank e gera a resposta.

Aqui o sistema consulta um banco vetorial usando a pergunta `query_for_search`. O método `similarity_search` retorna os k documentos cujo embedding é mais parecido com o embedding da pergunta. Ou seja, é uma busca semântica: os documentos não precisam conter exatamente as mesmas palavras, mas precisam ter significado semelhante.

Logo em seguida, ocorre um reranqueamento. Os documentos recuperados na busca vetorial são reavaliados por um modelo de ranking (aqui, CrossEncoder) que analisa diretamente o par (pergunta, documento) para estimar o quão relevante aquele documento é para responder à pergunta. Após essa avaliação, os documentos são ordenados por relevância e apenas os top_k melhores são mantidos. E então, com a formulação de um contexto apropriado, o modelo de linguagem é chamado para retornar a resposta.

In [ ]:
def ask_question(query, vector_store, k=10, top_k=5, translate=True):
    if vector_store is None:
        # Não traduz nem consulta o índice no baseline sem recuperação.
        return generate_answer(query, [], use_rag=False), []
    query_for_search = deepl_translate(query) if translate else query
    docs = vector_store.similarity_search(query_for_search, k=k)
    reranked = rerank_documents(query_for_search, docs, top_k=top_k)
    answer = generate_answer(query, reranked)
    return answer, reranked

In [ ]:
# Exemplo de execução, antes de chamar o modelo:
query = "Qual o prazo para o pagamento da taxa de inscrição das equipes de F1?"
vector_store, k, top_k, translate = vector_store_b, 2, 1, True

# Tradução para alinhar idioma da pergunta com idioma da base de dados
query_for_search = deepl_translate(query) if translate else query
print(query_for_search)

# Busca por similaridade
docs = vector_store.similarity_search(query_for_search, k=k)
print(docs)

# Resultado da busca semântica ordenado por relevância para a pergunta
reranked = rerank_documents(query_for_search, docs, top_k=top_k)
print(reranked)

# 12. Execução rápida (exemplo)

Teste uma pergunta com cada metodo para comparar qualitativamente.

In [ ]:
# Exemplo rapido
question = "Qual o prazo para o pagamento da taxa de inscrição das equipes de F1?"
resposta_a, _ = ask_question(question, vector_store_a)
resposta_b, _ = ask_question(question, vector_store_b)
resposta_c, _ = ask_question(question, vector_store_c)
print("Metodo A:", resposta_a)
print("Metodo B:", resposta_b)
print("Metodo C:", resposta_c)

# 13. Referências de resposta e juízes LLM

O trabalho utilizou um piloto de 10 perguntas e um conjunto de 100 perguntas sintéticas, com revisão manual, para explorar os métodos de recuperação. As respostas esperadas estão em `filesIAColab/tests.json`.

A avaliação usa prompts inspirados no CCRS para coerência contextual, relevância da resposta, densidade de informação, corretude e recall. Cada juiz deve retornar um inteiro entre 0 e 100. Saídas inválidas tornam-se valores ausentes, e não notas negativas.

Juízes LLM reduzem o trabalho manual, mas não eliminam subjetividade ou vieses. As notas não são porcentagens de acerto. A configuração dos modelos históricos não está registrada nos CSVs; o juiz abaixo usa `NVIDIA_JUDGE_MODEL` da configuração local.

O teste histórico sem RAG reutilizava o prompt que exigia contexto. A execução atual usa um prompt independente, portanto seus resultados devem ser tratados como um novo experimento. Coerência contextual e o score global que a inclui não são diretamente comparáveis sem contexto.

In [ ]:
from prompts.llm_judges_prompts import (
    cc_prompt,
    qr_prompt,
    id_prompt,
    ac_prompt,
    ir_prompt,
)

llm_judge = ChatNVIDIA(
    model=NVIDIA_JUDGE_MODEL,
    api_key=os.environ.get("NVIDIA_API_KEY", ""),
    temperature=0,
    top_p=0.7,
    max_completion_tokens=2048,
)

cc_judge = cc_prompt | llm_judge
qr_judge = qr_prompt | llm_judge
id_judge = id_prompt | llm_judge
ac_judge = ac_prompt | llm_judge
ir_judge = ir_prompt | llm_judge


def evaluate_response(judge, question, context, response, ground_truth):
    judge_response = judge.invoke({
        "question": question,
        "context": context,
        "response": response,
        "ground_truth": ground_truth,
    })
    return judge_response.content

# 14. Avaliação em CSV

Executa perguntas e juízes para uma faixa do conjunto de referência. Cada lote gera um CSV e um arquivo `.metadata.json` com a configuração da execução. Os resultados podem variar entre chamadas; os metadados não garantem determinismo.

In [ ]:
import json
import pandas as pd
import re

def safe_int(value):
    match = re.fullmatch(r"(?:100|[1-9]?\d)(?:\.0+)?", str(value).strip())
    return int(float(match.group())) if match else float("nan")


def evaluate_with_judges(question, context, answer, ground_truth):
    scores = {}

    scores['cc_judge'] = safe_int(
        cc_judge.invoke({
            "context": context,
            "response": answer
        }).content
    )
    scores['qr_judge'] = safe_int(
        qr_judge.invoke({
            "question": question,
            "response": answer
        }).content
    )
    scores['id_judge'] = safe_int(
        id_judge.invoke({
            "question": question,
            "context": context,
            "response": answer
        }).content
    )
    scores['ac_judge'] = safe_int(
        ac_judge.invoke({
            "context": context,
            "response": answer,
            "ground_truth": ground_truth
        }).content
    )
    scores['ir_judge'] = safe_int(
        ir_judge.invoke({
            "context": context,
            "response": answer,
            "ground_truth": ground_truth
        }).content
    )
    return scores


def process_json_questions_range(
    json_file_path,
    output_csv_path,
    vector_store,
    k=10,
    top_k=5,
    start_idx=1,
    end_idx=None,
):
    """
    start_idx e end_idx sao 1-based e inclusivos.
    Exemplo: start_idx=1, end_idx=20.
    """
    from datetime import datetime, timezone
    import hashlib
    from importlib.metadata import version

    started_at = datetime.now(timezone.utc).isoformat()
    with open(json_file_path, 'r', encoding='utf-8') as f:
        json_data = json.load(f)

    total = len(json_data)
    if end_idx is None or end_idx > total:
        end_idx = total
    if start_idx < 1 or start_idx > total:
        raise ValueError(f"start_idx fora do range (1..{total})")
    if end_idx < start_idx:
        raise ValueError("end_idx deve ser >= start_idx")

    results_data = []
    slice_data = json_data[start_idx - 1 : end_idx]
    range_total = end_idx - start_idx + 1

    for i, item in enumerate(slice_data, start=start_idx):
        pergunta = item.get('pergunta', '')
        resposta_esperada = item.get('resposta_esperada', '')

        answer, retrieved = ask_question(pergunta, vector_store, k=k, top_k=top_k)
        context_text = "\n\n".join([doc.page_content for doc in retrieved])
        scores = evaluate_with_judges(
            pergunta, context_text, answer, resposta_esperada
        )

        result_row = {
            'Pergunta': pergunta,
            'Resposta_Esperada': resposta_esperada,
            'Resposta_Gerada': answer,
            'Contexto_Recuperado': context_text,
            'Context_Coherence_Score': scores.get('cc_judge', 0),
            'Question_Relevance_Score': scores.get('qr_judge', 0),
            'Information_Density_Score': scores.get('id_judge', 0),
            'Answer_Correctness_Score': scores.get('ac_judge', 0),
            'Information_Recall_Score': scores.get('ir_judge', 0),
        }
        results_data.append(result_row)

        print(f"[{i - start_idx + 1}/{range_total}] OK (pergunta {i})")

    df_results = pd.DataFrame(results_data)
    df_results.to_csv(output_csv_path, index=False, encoding='utf-8')
    metadata = {
        "started_at_utc": started_at,
        "completed_at_utc": datetime.now(timezone.utc).isoformat(),
        "generation_model": NVIDIA_MODEL,
        "judge_model": NVIDIA_JUDGE_MODEL,
        "temperature": 0.5,
        "top_p": 0.7,
        "max_completion_tokens": 2048,
        "judge_temperature": 0,
        "judge_top_p": 0.7,
        "judge_max_completion_tokens": 2048,
        "retrieval_enabled": vector_store is not None,
        "k": k,
        "top_k": top_k,
        "start_idx": start_idx,
        "end_idx": end_idx,
        "data_revision": DATA_ACTUAL_REVISION,
        "questions_sha256": hashlib.sha256(Path(json_file_path).read_bytes()).hexdigest(),
        "notebook_sha256": hashlib.sha256((PROJECT_ROOT / "FIA_Regulation_RAG.ipynb").read_bytes()).hexdigest(),
        "embedding_model": "sentence-transformers/all-MiniLM-L6-v2",
        "reranker": "cross-encoder/ms-marco-MiniLM-L-6-v2",
        "python_version": sys.version.split()[0],
        "packages": {name: version(name) for name in (
            "langchain-core", "langchain-nvidia-ai-endpoints", "sentence-transformers", "faiss-cpu",
        )},
        "prompt_version": "portfolio-independent-no-rag-v1",
    }
    Path(output_csv_path).with_suffix(".metadata.json").write_text(
        json.dumps(metadata, ensure_ascii=False, indent=2) + "\n", encoding="utf-8",
    )
    print(f"Resultados salvos em {output_csv_path}")
    return df_results

In [ ]:
if RUN_EVALUATIONS:
    # Exemplo de uso
    df_eval = process_json_questions_range(
        json_file_path="filesIAColab/tests.json",
        output_csv_path="avaliacao_range.csv",
        vector_store=vector_store_b,
        start_idx=1,
        end_idx=3,
    )
else:
    print("Avaliação ignorada. Defina RUN_EVALUATIONS = True para executar.")

# 15. Análise comparativa entre os resultados

Vamos rodar todos os dados de teste; que consiste de 100 perguntas elaboradas com o auxílio do Gemini chat e sanitização manual.

In [ ]:
if RUN_EVALUATIONS:
    # Avaliação para Método sem RAG
    print(">> Avaliando Método Sem RAG")
    df_eval = process_json_questions_range(
        json_file_path="filesIAColab/tests.json",
        output_csv_path="avaliacao_noRAG.csv",
        vector_store=None,
        k=40,
        top_k=8,
        start_idx=1,
        end_idx=100,
    )
else:
    print("Avaliação ignorada. Defina RUN_EVALUATIONS = True para executar.")

In [ ]:
if RUN_EVALUATIONS:
    # Avaliação para Método A
    print(">> Avaliando Método A")
    df_eval = process_json_questions_range(
        json_file_path="filesIAColab/tests.json",
        output_csv_path="avaliacao_a.csv",
        vector_store=vector_store_a,
        k=60,
        top_k=8,
        start_idx=1,
        end_idx=100,
    )
else:
    print("Avaliação ignorada. Defina RUN_EVALUATIONS = True para executar.")

In [ ]:
if RUN_EVALUATIONS:
    # Avaliação para Método B
    print(">> Avaliando Método B")
    df_eval = process_json_questions_range(
        json_file_path="filesIAColab/tests.json",
        output_csv_path="avaliacao_b.csv",
        vector_store=vector_store_b,
        k=80,
        top_k=10,
        start_idx=1,
        end_idx=100,
    )
else:
    print("Avaliação ignorada. Defina RUN_EVALUATIONS = True para executar.")

In [ ]:
if RUN_EVALUATIONS:
    # Avaliação para Método C
    print(">> Avaliando Método C")
    df_eval = process_json_questions_range(
        json_file_path="filesIAColab/tests.json",
        output_csv_path="avaliacao_c.csv",
        vector_store=vector_store_c,
        k=60,
        top_k=8,
        start_idx=1,
        end_idx=100,
    )
else:
    print("Avaliação ignorada. Defina RUN_EVALUATIONS = True para executar.")

# 16. Carregar novas avaliações

Carrega os quatro CSVs gerados pelas células de avaliação acima. Se faltarem arquivos, a análise é ignorada. Os resultados históricos publicados ficam separados em `results/historical/`; consulte `results/README.md` para proveniência e limitações.

In [ ]:
import pandas as pd

csv_paths = {
    "A": PROJECT_ROOT / "avaliacao_a.csv",
    "B": PROJECT_ROOT / "avaliacao_b.csv",
    "C": PROJECT_ROOT / "avaliacao_c.csv",
    "noRAG": PROJECT_ROOT / "avaliacao_noRAG.csv",
}
missing_csvs = [path.name for path in csv_paths.values() if not path.is_file()]
ANALYSIS_READY = not missing_csvs
if ANALYSIS_READY:
    df_a, df_b, df_c, df_norag = [pd.read_csv(path) for path in csv_paths.values()]
    if not df_a["Pergunta"].is_unique:
        raise ValueError("Perguntas duplicadas no método A; revise os arquivos antes de comparar.")
    for frame in [df_b, df_c, df_norag]:
        if not frame["Pergunta"].reset_index(drop=True).equals(df_a["Pergunta"].reset_index(drop=True)):
            raise ValueError("Perguntas ou ordem diferentes entre métodos; comparação interrompida.")
    for name, df in zip(csv_paths, [df_a, df_b, df_c, df_norag]):
        print(f"Shape método {name}: {df.shape}")
    display(df_a.head(15))
else:
    print("Análise ignorada. Execute as avaliações para gerar: " + ", ".join(missing_csvs))

# 17. Definir as métricas avaliadas

Aqui definimos as colunas que representam os scores dados pelos LLM judges.

In [ ]:
if ANALYSIS_READY:
    score_columns = [
        "Context_Coherence_Score",
        "Question_Relevance_Score",
        "Information_Density_Score",
        "Answer_Correctness_Score",
        "Information_Recall_Score"
    ]

Garantimos que todos os valores são numéricos:

In [ ]:
if ANALYSIS_READY:
    for col in score_columns:
        df_a[col] = pd.to_numeric(df_a[col], errors="coerce").where(lambda scores: scores.between(0, 100))
        df_b[col] = pd.to_numeric(df_b[col], errors="coerce").where(lambda scores: scores.between(0, 100))
        df_c[col] = pd.to_numeric(df_c[col], errors="coerce").where(lambda scores: scores.between(0, 100))
        df_norag[col] = pd.to_numeric(df_norag[col], errors="coerce").where(lambda scores: scores.between(0, 100))
    for name, frame in [("A", df_a), ("B", df_b), ("C", df_c), ("noRAG", df_norag)]:
        print(f"Notas ausentes/inválidas ({name}): {frame[score_columns].isna().sum().sum()}")

# 18. Média das métricas

Descrição:
Esta análise calcula a média de cada métrica para cada método de RAG.
Isso permite entender em quais critérios cada sistema é melhor.

In [ ]:
if ANALYSIS_READY:
    mean_a = df_a[score_columns].mean()
    mean_b = df_b[score_columns].mean()
    mean_c = df_c[score_columns].mean()
    mean_norag = df_norag[score_columns].mean()

    comparison = pd.DataFrame({
        "Metodo_A": mean_a,
        "Metodo_B": mean_b,
        "Metodo_C": mean_c,
        "Metodo_noRAG": mean_norag
    })

    comparison

# 19. Calcular um score global do RAG

Descrição:
Criamos um score único combinando todas as métricas.
A correção factual recebe peso maior porque é a métrica mais importante.

- Pesos utilizados:
  - 40% correção factual
  - 20% relevância
  - 20% recall
  - 10% coerência
  - 10% densidade

In [ ]:
if ANALYSIS_READY:
    def compute_global_score(df):
        return (
            df["Answer_Correctness_Score"] * 0.4 +
            df["Question_Relevance_Score"] * 0.2 +
            df["Information_Recall_Score"] * 0.2 +
            df["Context_Coherence_Score"] * 0.1 +
            df["Information_Density_Score"] * 0.1
        )

    df_a["Global_Score"] = compute_global_score(df_a)
    df_b["Global_Score"] = compute_global_score(df_b)
    df_c["Global_Score"] = compute_global_score(df_c)
    df_norag["Global_Score"] = compute_global_score(df_norag)

    print("Score global médio método A:", df_a["Global_Score"].mean())
    print("Score global médio método B:", df_b["Global_Score"].mean())
    print("Score global médio método C:", df_c["Global_Score"].mean())
    print("Score global médio método noRAG:", df_norag["Global_Score"].mean())

# 20. Comparação pergunta a pergunta

Descrição:
Aqui verificamos quantas perguntas cada método venceu.

Isso é comum em avaliações de RAG em papers.

In [ ]:
if ANALYSIS_READY:
    wins_a = (df_a["Global_Score"] > df_b["Global_Score"]).sum()
    wins_b = (df_b["Global_Score"] > df_a["Global_Score"]).sum()

    print("Método A venceu:", wins_a)
    print("Método B venceu:", wins_b)

# 21. Gráfico de comparação das métricas

Descrição:
Visualização comparando os dois métodos em cada métrica.

In [ ]:
if ANALYSIS_READY:
    import matplotlib.pyplot as plt

    comparison.plot(kind="bar", figsize=(10,6))

    plt.title("Comparação das métricas de avaliação do RAG")
    plt.ylabel("Score médio")
    plt.xticks(rotation=45)

    plt.grid(axis="y")

    plt.show()

# 22. Distribuição do score global

Descrição:
Mostra a estabilidade dos dois métodos.

In [ ]:
if ANALYSIS_READY:
    plt.figure(figsize=(8,5))

    plt.hist(df_a["Global_Score"], alpha=0.5, label="Metodo A")
    plt.hist(df_b["Global_Score"], alpha=0.5, label="Metodo B")
    plt.hist(df_c["Global_Score"], alpha=0.5, label="Metodo C")
    plt.hist(df_norag["Global_Score"], alpha=0.5, label="Metodo NoRAG")

    plt.xlabel("Score Global")
    plt.ylabel("Frequência")

    plt.title("Distribuição dos scores de qualidade do RAG")

    plt.legend()

    plt.show()

In [ ]:
# Backup local dos resultados (substitui a montagem do Google Drive no Colab).
import shutil

backup_dir = PROJECT_ROOT / "backups" / "avaliacoes"
result_files = sorted(PROJECT_ROOT.glob("avaliacao*.csv"))
if result_files:
    backup_dir.mkdir(parents=True, exist_ok=True)
    for result_file in result_files:
        shutil.copy2(result_file, backup_dir / result_file.name)
        print(f"Copiado: {result_file.name} -> {backup_dir}")
else:
    print("Nenhum CSV de avaliação foi gerado ainda.")